# Binomial Stock Dynamics & One-Period Option Pricing

This notebook develops the binomial stock model from its financial-mathematical foundations and extends it to **one-period European option pricing**.

**Underlying used for empirical calibration**

- **TotalEnergies SE ADR (`TTE`)** — oil & gas equity listed in the U.S.

$$
\text{Discounting}
\rightarrow
\text{Binomial Dynamics}
\rightarrow
\text{Calibration}
\rightarrow
\text{Martingales}
\rightarrow
\text{Risk-Neutral Measure}
\rightarrow
\text{Option Payoffs}
\rightarrow
\text{Replication}
\rightarrow
\text{No-Arbitrage Pricing}
$$

The purpose is not to claim that a two-state model describes markets perfectly. The purpose is to derive the core pricing logic transparently.

## 1. Financial foundation: value through time

A future payoff cannot be compared directly with money held today. With a one-period risk-free rate $r$, a deterministic future payoff $F$ has present value

$$
P=\frac{F}{1+r}.
$$

For $N$ periods,

$$
P=\frac{F}{(1+r)^N}.
$$

This same idea will later be applied to a stock-price process. Instead of discounting one deterministic cash flow, we will discount a **random asset price** by the risk-free account.

The key financial principle is **no arbitrage**: if two portfolios generate exactly the same future payoff, they cannot have different prices today without creating a risk-free profit opportunity.

## 2. The binomial stock model

Let $S_n$ denote the stock price at time $n$. In the binomial model,

$$
S_{n+1}=H_{n+1}S_n,
$$

where the one-step gross return $H_{n+1}$ has only two possible values:

$$
H_{n+1}=\begin{cases}
u, & \text{with probability }p,\\
d, & \text{with probability }1-p.
\end{cases}
$$

Therefore after one step the stock is either $uS_n$ or $dS_n$. Over many periods,

$$
S_N=S_0\prod_{i=1}^{N}H_i.
$$

Under the model assumptions the $H_i$ are independent and identically distributed. This produces a recombining tree when $ud=1$.

### Information structure — only what we need

We denote by

$$
\mathcal F_n=\sigma(H_1,\ldots,H_n)
$$

the information generated by all stock moves observed up to time $n$. The sequence

$$
\mathcal F_0\subseteq\mathcal F_1\subseteq\cdots\subseteq\mathcal F_n
$$

is a **filtration**: information accumulates over time.

The conditional expectation

$$
E[X\mid\mathcal F_n]
$$

is the best mean-square predictor of a future random quantity $X$ using only the information available at time $n$.

For the binomial model,

$$
E[S_{n+1}\mid\mathcal F_n]
= S_n\big(pu+(1-p)d\big).
$$

Only the current price $S_n$ is needed for the next-step conditional expectation. This is the **Markov property** of the model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

TRADING_DAYS = 252
TICKER = "TTE"
START_DATE = "2018-01-01"
END_DATE = "2026-09-13"

# Illustrative model input only; not presented as the live market risk-free rate.
ANNUAL_RISK_FREE_RATE = 0.03

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


def log_returns(prices):
    return np.log(prices / prices.shift(1))


def annualized_log_return_stats(prices, periods_per_year=252):
    r = log_returns(prices).dropna()
    mu = periods_per_year * r.mean()
    sigma = np.sqrt(periods_per_year) * r.std(ddof=1)
    return float(mu), float(sigma)


def calibrate_binomial(mu, sigma, dt):
    u = np.exp(sigma * np.sqrt(dt))
    d = np.exp(-sigma * np.sqrt(dt))
    p = 0.5 + 0.5 * (mu / sigma) * np.sqrt(dt) if sigma > 0 else 0.5
    return float(u), float(d), float(p)


def step_risk_free_rate(annual_rate, dt):
    return float((1.0 + annual_rate) ** dt - 1.0)


def risk_neutral_probability(u, d, r_step):
    return float(((1.0 + r_step) - d) / (u - d))


def no_arbitrage_condition(u, d, r_step):
    return bool(d < (1.0 + r_step) < u)


def simulate_paths(s0, u, d, prob_up, n_steps, n_paths, seed=42):
    rng = np.random.default_rng(seed)
    up_moves = rng.random((n_paths, n_steps)) < prob_up
    gross_returns = np.where(up_moves, u, d)
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = s0
    paths[:, 1:] = s0 * np.cumprod(gross_returns, axis=1)
    return paths


def discount_paths(paths, r_step):
    steps = np.arange(paths.shape[1])
    return paths / ((1.0 + r_step) ** steps)

## 3. Empirical calibration asset: TotalEnergies

The empirical experiment uses **TotalEnergies (`TTE`)** rather than a generic classroom stock.

The daily log return is

$$
r_t
=
\ln\left(\frac{S_t}{S_{t-1}}\right).
$$

We estimate

$$
\hat{\mu}
=
252\,\bar r,
$$

and

$$
\hat{\sigma}
=
\sqrt{252}\,s_r.
$$

These quantities calibrate the physical binomial dynamics. They are not the risk-neutral pricing probabilities.

In [ ]:
import yfinance as yf

raw = yf.download(
    TICKER,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False,
)

if raw.empty:
    raise RuntimeError(
        "No TotalEnergies data were downloaded. Run the notebook with internet access."
    )

if isinstance(raw.columns, pd.MultiIndex):
    close = raw["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
else:
    close = raw["Close"]

prices = close.rename(TICKER).dropna().to_frame()
prices.head()

In [ ]:
ax = prices.plot(figsize=(10, 5), linewidth=1.8, legend=False)
ax.set_title("TotalEnergies (TTE) — Adjusted Closing Price")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
ax.grid(alpha=0.25)
plt.show()

In [ ]:
returns = prices.apply(log_returns)

ax = returns.plot(figsize=(10, 5), alpha=0.8, legend=False)
ax.set_title("TotalEnergies (TTE) — Daily Log Returns")
ax.set_xlabel("Date")
ax.set_ylabel("Log return")
ax.grid(alpha=0.25)
plt.show()

### Empirical return distribution

The binomial model allows only two one-period gross returns, $u$ and $d$. Real equity returns are continuous and can contain asymmetry, heavy tails and extreme observations.

The goal is therefore not to claim that real returns are literally binomial, but to calibrate a transparent discrete model and use it to study asset-pricing principles.

In [ ]:
vals = returns[TICKER].dropna()

plt.figure(figsize=(8, 4.5))
plt.hist(vals, bins=40, density=True, alpha=0.65, label="Empirical log returns")

x = np.linspace(vals.min(), vals.max(), 400)
plt.plot(
    x,
    stats.norm.pdf(x, vals.mean(), vals.std(ddof=1)),
    linewidth=2,
    label="Normal reference",
)

plt.title("TTE — Empirical Daily Log Returns")
plt.xlabel("Log return")
plt.ylabel("Density")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 4. Calibrating the binomial model

For a horizon split into intervals of length $\Delta t$, the course scaling uses

$$
u=d^{-1}=\exp(\sigma\sqrt{\Delta t}).
$$

Thus

$$
d=\exp(-\sigma\sqrt{\Delta t}).
$$

The physical up probability can be approximated by

$$
p\approx \frac12+\frac12\frac{\mu}{\sigma}\sqrt{\Delta t}.
$$

Here $\mu$ and $\sigma$ are estimated from historical log returns. This $p$ belongs to the **physical/historical probability measure $P$**.

In [ ]:
dt = 1 / TRADING_DAYS

s = prices[TICKER].dropna()
mu_hat, sigma_hat = annualized_log_return_stats(s, periods_per_year=TRADING_DAYS)

u, d, p = calibrate_binomial(mu_hat, sigma_hat, dt)
r_step = step_risk_free_rate(ANNUAL_RISK_FREE_RATE, dt)
q = risk_neutral_probability(u, d, r_step)

calibration = pd.Series({
    "mu_hat": mu_hat,
    "sigma_hat": sigma_hat,
    "u": u,
    "d": d,
    "p_physical": p,
    "r_step": r_step,
    "q_risk_neutral": q,
    "no_arbitrage": no_arbitrage_condition(u, d, r_step),
}, name=TICKER)

calibration.to_frame("value")

## 5. Discounted prices and martingales

Define the discounted stock price

$$
S_n^*=\frac{S_n}{(1+r)^n}.
$$

Then

$$
E[S_{n+1}^*\mid\mathcal F_n]
=
\frac{pu+(1-p)d}{1+r}S_n^*.
$$

Therefore:

- if $pu+(1-p)d>1+r$, the discounted process is a **submartingale**;
- if $pu+(1-p)d<1+r$, it is a **supermartingale**;
- if $pu+(1-p)d=1+r$, it is a **martingale**.

A martingale satisfies

$$
E[S_{n+1}^*\mid\mathcal F_n]=S_n^*.
$$

Financially, under the appropriate probability measure, the discounted asset has no predictable excess gain relative to the risk-free asset.

## 6. Risk-neutral probability and no arbitrage

Instead of forcing the historical probability $p$ to satisfy the martingale condition, define another probability $q$ such that

$$
q u+(1-q)d=1+r.
$$

Solving for $q$ gives

$$
\boxed{q=\frac{1+r-d}{u-d}}.
$$

Under the corresponding measure $Q$,

$$
E_Q[S_{n+1}^*\mid\mathcal F_n]=S_n^*.
$$

For $Q$ to assign positive probability to both up and down states we require

$$
0<q<1,
$$

which is equivalent to the no-arbitrage condition

$$
\boxed{d<1+r<u}.
$$

### Important distinction

- $p$: linked to empirical/historical stock dynamics.
- $q$: selected so discounted prices are martingales and no-arbitrage pricing is possible.

The risk-neutral probability is therefore **not** a forecast that the stock will rise with probability $q$ in the real world.

In [ ]:
N_STEPS = 252
N_PATHS = 5000

s0 = float(prices[TICKER].iloc[0])
r_step_mc = step_risk_free_rate(ANNUAL_RISK_FREE_RATE, 1 / N_STEPS)

paths_q = simulate_paths(
    s0=s0,
    u=float(calibration["u"]),
    d=float(calibration["d"]),
    prob_up=float(calibration["q_risk_neutral"]),
    n_steps=N_STEPS,
    n_paths=N_PATHS,
    seed=42,
)

discounted_q = discount_paths(paths_q, r_step_mc)
mean_discounted = discounted_q.mean(axis=0)

martingale_check = pd.Series({
    "S0": s0,
    "mean_discounted_terminal": mean_discounted[-1],
    "relative_error": (mean_discounted[-1] - s0) / s0,
})

martingale_check.to_frame("value")

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(mean_discounted, label=r"Monte Carlo mean of $S_n^*$")
plt.axhline(s0, linestyle="--", label=r"Initial price $S_0$")
plt.title("TTE — Discounted-Price Martingale Check under Q")
plt.xlabel("Step")
plt.ylabel("Discounted price")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

The Monte Carlo average is not expected to equal $S_0$ exactly because only finitely many paths are simulated. As the number of simulated paths increases, the sampling error should shrink.

## 7. European options in one period

A derivative is a financial claim whose payoff depends on another asset.

For a European call with strike $K$,

$$
C_1=\max(S_1-K,0).
$$

For a European put,

$$
P_1=\max(K-S_1,0).
$$

In the one-period binomial model,

$$
S_1=
\begin{cases}
uS_0, & \text{up state},\\
dS_0, & \text{down state}.
\end{cases}
$$

Hence a derivative payoff is summarized by

$$
F_u=F(uS_0),
\qquad
F_d=F(dS_0).
$$

In [ ]:
S0_OPTION = float(prices[TICKER].iloc[-1])

u_opt = float(calibration["u"])
d_opt = float(calibration["d"])
r_opt = float(calibration["r_step"])
q_opt = float(calibration["q_risk_neutral"])

# At-the-money strike for a transparent numerical experiment.
K = S0_OPTION

S_up = u_opt * S0_OPTION
S_down = d_opt * S0_OPTION

call_up = max(S_up - K, 0.0)
call_down = max(S_down - K, 0.0)

put_up = max(K - S_up, 0.0)
put_down = max(K - S_down, 0.0)

states = pd.DataFrame({
    "state": ["Up", "Down"],
    "stock_price": [S_up, S_down],
    "call_payoff": [call_up, call_down],
    "put_payoff": [put_up, put_down],
}).set_index("state")

print(f"S0 = {S0_OPTION:.4f}")
print(f"K  = {K:.4f}")
print(f"q  = {q_opt:.6f}")
states

## 8. Risk-neutral option valuation

The risk-neutral probability satisfies

$$
qu+(1-q)d=1+r,
$$

so

$$
\boxed{
q=\frac{1+r-d}{u-d}
}.
$$

For any one-period derivative,

$$
\boxed{
V_0
=
\frac{qF_u+(1-q)F_d}{1+r}
}.
$$

Equivalently,

$$
V_0=E_Q[(1+r)^{-1}F].
$$

The historical probability $p$ does not enter the pricing formula.

In [ ]:
def risk_neutral_option_price(payoff_up, payoff_down, q, r_step):
    return (q * payoff_up + (1 - q) * payoff_down) / (1 + r_step)


call_price_q = risk_neutral_option_price(call_up, call_down, q_opt, r_opt)
put_price_q = risk_neutral_option_price(put_up, put_down, q_opt, r_opt)

pd.Series({
    "call_fair_value": call_price_q,
    "put_fair_value": put_price_q,
}).to_frame("risk_neutral_price")

## 9. Replicating portfolio

Construct a portfolio with $\theta_0$ in the risk-free asset and $\theta_1$ shares of stock:

$$
\theta_0(1+r)+\theta_1uS_0=F_u,
$$

$$
\theta_0(1+r)+\theta_1dS_0=F_d.
$$

Subtracting,

$$
\boxed{
\theta_1
=
\frac{F_u-F_d}{S_0(u-d)}
}.
$$

Then,

$$
\boxed{
\theta_0
=
\frac{F_u-\theta_1uS_0}{1+r}
}.
$$

The replication cost is

$$
V_0=\theta_0+\theta_1S_0.
$$

In [ ]:
def replicating_portfolio(S0, u, d, r_step, payoff_up, payoff_down):
    theta1 = (payoff_up - payoff_down) / (S0 * (u - d))
    theta0 = (payoff_up - theta1 * u * S0) / (1 + r_step)
    value0 = theta0 + theta1 * S0
    return theta0, theta1, value0


call_theta0, call_theta1, call_price_rep = replicating_portfolio(
    S0_OPTION, u_opt, d_opt, r_opt, call_up, call_down
)

put_theta0, put_theta1, put_price_rep = replicating_portfolio(
    S0_OPTION, u_opt, d_opt, r_opt, put_up, put_down
)

pd.DataFrame({
    "Call": [call_theta0, call_theta1, call_price_rep, call_price_q],
    "Put": [put_theta0, put_theta1, put_price_rep, put_price_q],
}, index=["theta_0", "theta_1", "replication_cost", "risk_neutral_price"])

### Replication check

For a perfectly replicated claim,

$$
\text{replication cost}
=
\text{risk-neutral value}.
$$

The replicating portfolio must also reproduce the derivative payoff in both states.

In [ ]:
call_rep_up = call_theta0 * (1 + r_opt) + call_theta1 * S_up
call_rep_down = call_theta0 * (1 + r_opt) + call_theta1 * S_down

put_rep_up = put_theta0 * (1 + r_opt) + put_theta1 * S_up
put_rep_down = put_theta0 * (1 + r_opt) + put_theta1 * S_down

pd.DataFrame({
    "option": ["Call", "Call", "Put", "Put"],
    "state": ["Up", "Down", "Up", "Down"],
    "option_payoff": [call_up, call_down, put_up, put_down],
    "replicating_portfolio_payoff": [
        call_rep_up, call_rep_down, put_rep_up, put_rep_down
    ],
})

## 10. Mispricing and arbitrage

Let $V_0$ be the replication value.

If

$$
M_0>V_0,
$$

short the option and buy the replicating portfolio.

If

$$
M_0<V_0,
$$

buy the option and short the replicating portfolio.

Because the terminal payoffs cancel state by state, the only price compatible with no arbitrage is

$$
\boxed{M_0=V_0}.
$$

In [ ]:
market_prices = pd.Series({
    "10% below fair value": 0.90 * call_price_q,
    "fair value": call_price_q,
    "10% above fair value": 1.10 * call_price_q,
})

def arbitrage_interpretation(market_price, fair_value, tol=1e-12):
    if market_price > fair_value + tol:
        return "Short option + buy replicating portfolio"
    if market_price < fair_value - tol:
        return "Buy option + short replicating portfolio"
    return "No arbitrage from relative mispricing"

arbitrage_table = pd.DataFrame({"market_option_price": market_prices})
arbitrage_table["fair_value"] = call_price_q
arbitrage_table["locked_initial_difference"] = (
    arbitrage_table["market_option_price"] - call_price_q
).abs()
arbitrage_table["strategy"] = [
    arbitrage_interpretation(x, call_price_q)
    for x in arbitrage_table["market_option_price"]
]

arbitrage_table

## 11. Superhedging and complete markets

Let $P_S$ be the seller's minimum superhedging price and $P_B$ the buyer's maximum compatible price.

In general,

$$
P_B
\le
E_Q[F^*]
\le
P_S.
$$

In the one-period binomial market the contingent claim is exactly replicable, so

$$
\boxed{
P_B=P_S=E_Q[F^*]
}.
$$

Thus the one-period binomial market is **complete**, and the equivalent martingale measure is unique.

In [ ]:
terminal_grid = np.linspace(0.6 * S0_OPTION, 1.4 * S0_OPTION, 400)

plt.figure(figsize=(9, 5))
plt.plot(terminal_grid, np.maximum(terminal_grid - K, 0), label="Call payoff")
plt.plot(terminal_grid, np.maximum(K - terminal_grid, 0), label="Put payoff")
plt.axvline(K, linestyle="--", linewidth=1, label="Strike K")
plt.title("European Option Payoffs at Expiry")
plt.xlabel(r"Terminal stock price $S_1$")
plt.ylabel("Payoff")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 12. Physical-measure simulations

Under the historical measure $P$, the up probability is the calibrated $p$ rather than $q$. This section visualizes possible one-year paths implied by the empirical drift and volatility.

The objective is not point forecasting. It is to understand the distribution of paths implied by the model assumptions.

In [ ]:
for asset in prices.columns:
    row = calibration
    s0 = float(prices[asset].dropna().iloc[0])
    p = float(row["p_physical"])

    if not (0 <= p <= 1):
        print(f"{asset}: calibrated p={p:.4f} is outside [0,1]; physical simulation skipped.")
        continue

    paths_p = simulate_paths(
        s0=s0,
        u=float(row["u"]),
        d=float(row["d"]),
        prob_up=p,
        n_steps=252,
        n_paths=250,
        seed=7,
    )

    plt.figure(figsize=(10, 5))
    plt.plot(paths_p[:50].T, alpha=0.35)
    plt.title(f"{asset}: calibrated binomial paths under P")
    plt.xlabel("Trading step")
    plt.ylabel("Price")
    plt.grid(alpha=0.2)
    plt.show()

## 13. From the binomial model to a lognormal limit

Because

$$
\ln\left(\frac{S_N}{S_0}\right)
=\sum_{i=1}^{N}\ln H_i,
$$

the terminal log return is a sum of independent random variables. The Central Limit Theorem therefore motivates a Gaussian approximation for the log return when $N$ is large.

With the appropriate scaling,

$$
\ln\left(\frac{S(t)}{S(0)}\right)
\sim N(\mu t,\sigma^2t).
$$

Equivalently,

$$
S(t)=S(0)\exp\left(\mu t+\sigma\sqrt{t}Z\right),
\qquad Z\sim N(0,1),
$$

so terminal prices have a lognormal structure. This provides the conceptual bridge from the discrete binomial model toward continuous-time stock models.

In [ ]:
def standardized_terminal_log_returns(mu, sigma, n_steps, n_paths=30000, seed=123):
    dt_local = 1 / n_steps
    u, d, p = calibrate_binomial(mu, sigma, dt_local)
    if not (0 <= p <= 1):
        return None
    paths = simulate_paths(1.0, u, d, p, n_steps, n_paths, seed=seed)
    terminal_log_return = np.log(paths[:, -1])
    return (terminal_log_return - terminal_log_return.mean()) / terminal_log_return.std(ddof=1)

asset = prices.columns[0]
mu = float(calibration["mu_hat"])
sigma = float(calibration["sigma_hat"])

for n_steps in [12, 52, 252]:
    z = standardized_terminal_log_returns(mu, sigma, n_steps)
    if z is None:
        continue
    x = np.linspace(-4, 4, 400)
    plt.figure(figsize=(8, 4.5))
    plt.hist(z, bins=45, density=True, alpha=0.65, label="standardized terminal log return")
    plt.plot(x, stats.norm.pdf(x), linewidth=2, label="N(0,1)")
    plt.title(f"{asset}: CLT convergence experiment, N={n_steps}")
    plt.xlabel("standardized log return")
    plt.ylabel("density")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

## 14. What this project demonstrates

1. **Discounting** moves value across time consistently.
2. **No arbitrage** restricts admissible asset prices.
3. The **binomial model** represents stock dynamics through up/down gross returns.
4. **Filtrations and conditional expectations** formalize evolving information.
5. **Discounted prices** lead to martingale concepts.
6. The historical probability $p$ and risk-neutral probability $q$ have different roles.
7. **European calls and puts** are contingent claims.
8. A derivative can be valued through an exact **replicating portfolio**.
9. **Risk-neutral valuation and replication coincide**.
10. Relative mispricing generates an **arbitrage** strategy.
11. Exact replicability gives equal buyer and seller prices.
12. The one-period binomial market is **complete** and has a unique equivalent martingale measure.
13. TotalEnergies data provide a real empirical calibration case.
14. The CLT experiment links repeated binomial dynamics to a lognormal limiting structure.

## 15. Current scope

This project now covers the one-period derivative-pricing logic introduced in the course:

- European call and put payoffs,
- risk-neutral valuation,
- exact replication,
- no-arbitrage pricing,
- superhedging bounds,
- complete-market interpretation.

It intentionally stops at the **one-period** model. Multi-period option trees and backward induction should be added only when the corresponding course material is introduced.
